In [ ]:

import sys
import os


IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    print("Running in Google Colab environment.")
    repo_dir = "/content/basic-llm"
    if not os.path.exists(repo_dir):
        print("Cloning repository into Colab...")
        !git clone https://github.com/Vishaal-sathya/basic-llm.git {repo_dir}
    else:
        print("Pulling latest updates into Colab...")
        !git -C {repo_dir} pull

    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    %cd {repo_dir}
else:
    print("Running in local environment.")
    repo_dir = os.path.abspath(".")
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)

print(f"Current working directory: {os.getcwd()}")
print(f"Python sys.path[0]: {sys.path[0]}")

Running in Google Colab environment.
Cloning repository into Colab...
Cloning into '/content/basic-llm'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 96 (delta 26), reused 85 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (96/96), 908.00 KiB | 37.83 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/basic-llm
Current working directory: /content/basic-llm
Python sys.path[0]: /content/basic-llm


In [ ]:

import torch
import tiktoken
from llm.llm_config import LLM_CONFIG
from llm.llm_architecture import LLMModel
from training.prepare_data import create_loaders
from training.train_model import train_model


print("LLM Configuration:", LLM_CONFIG)

All modules imported successfully!
LLM Configuration: {'vocab_size': 50256, 'context_length': 1024, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.1, 'qkv_bias': False}


In [ ]:

file_path = "the-verdict.txt"
train_ratio = 0.9


train_loader, val_loader = create_loaders(
    file_path=file_path,
    train_ratio=train_ratio,
    batch_size=2,
    max_length=256,
    stride=256
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Training batches: 9
Validation batches: 1


In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(123)
model = LLMModel(LLM_CONFIG)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total model parameters: {total_params:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
tokenizer = tiktoken.get_encoding('gpt2')

Using device: cuda
Total model parameters: 163,008,000


In [ ]:

num_epochs = 10

train_losses, val_losses, tokens_seen = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    num_epochs=num_epochs,
    eval_freq=5,
    eval_iter=1,
    start_context="Every effort moves you",
    tokenizer=tokenizer
)

Ep 1 (Step 000000): Train loss 10.128, Val loss 10.165
Ep 1 (Step 000005): Train loss 7.796, Val loss 8.340
Every effort moves you the had the, the to.I it of, that me ofis of. his him he of a      him a that it, the with my--.    ", he    he.,
Ep 2 (Step 000010): Train loss 6.790, Val loss 7.125
Ep 2 (Step 000015): Train loss 5.837, Val loss 6.583
Every effort moves you the my": he was he, it." it.  " the his.I".  is  "I-- my, it at the the the his that it:-- of the I G of of was you
Ep 3 (Step 000020): Train loss 15.988, Val loss 16.169
Ep 3 (Step 000025): Train loss 5.134, Val loss 6.353
Every effort moves you." " his--I was--'s-- had't-- with her, in in the of his own that my he it, and I, a was one of a and of I said his one.I that it, as of her, he
Ep 4 (Step 000030): Train loss 4.117, Val loss 6.362
Ep 4 (Step 000035): Train loss 4.272, Val loss 6.330
Every effort moves you of his that and my I felt to "Yes I said.  And have he was to--his. Rick a of it happened-- him was was ba